# 面试问题：怎样保证训练可复现，并做到断点续训后与不中断训练一致？

**一句话回答**：固定并隔离 Python、NumPy、PyTorch/各设备 RNG；保存模型、优化器、调度器、scaler、sampler/cursor、micro-step 与全部 RNG 状态；恢复时先重建同结构对象，再加载状态并从同一数据位置继续。可复现分“同机逐位”“同软件栈近似”“跨硬件统计一致”，不能只说设置 seed。

本 Notebook 用带 Dropout 的自定义网络做连续训练与恢复训练逐位对照，并覆盖 sampler、worker seed、半累积窗口和 checkpoint 完整性。

In [ ]:
import copy, hashlib, io, json, math, random
import numpy as np
import torch
from torch import nn

def seed_all102(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
SEED102=10201; seed_all102(SEED102)
x102=torch.randn(64,4); y102=(x102@torch.tensor([1.,-2.,.5,1.5])+.2).unsqueeze(1)
assert x102.shape==(64,4) and y102.shape==(64,1)
assert torch.isfinite(x102).all() and torch.isfinite(y102).all()
assert SEED102==10201

## 1. Seed 是起点，不是完整答案

Python、NumPy 和 PyTorch 各有 RNG；CUDA 还可能有每设备状态。固定 seed 只保证相同调用序列产生相同随机数，任何新增日志采样、不同 batch 顺序或提前返回都会改变后续随机流。任务级 Generator 能减少模块间耦合。

In [ ]:
seed_all102(77); seq_a102=(random.random(),np.random.rand(),torch.rand(3)); seed_all102(77); seq_b102=(random.random(),np.random.rand(),torch.rand(3))
gen_a102=torch.Generator().manual_seed(8); gen_b102=torch.Generator().manual_seed(8)
assert seq_a102[0]==seq_b102[0] and seq_a102[1]==seq_b102[1]
assert torch.equal(seq_a102[2],seq_b102[2])
assert torch.equal(torch.rand(5,generator=gen_a102),torch.rand(5,generator=gen_b102))

## 2. 数据顺序也是训练状态

随机 sampler 至少要保存 epoch 与当前 cursor，或保存 Generator 状态/本轮 permutation。只保存 epoch 会在 epoch 中间恢复时重复样本。下面的 sampler 用 `seed+epoch` 生成确定 permutation，并显式持久化 cursor。

In [ ]:
class Sampler102:
    def __init__(self,n,seed,epoch=0,cursor=0): self.n,self.seed,self.epoch,self.cursor=n,seed,epoch,cursor; self.order=self._order()
    def _order(self): return np.random.default_rng(self.seed+self.epoch).permutation(self.n)
    def take(self,k): out=self.order[self.cursor:min(self.cursor+k,self.n)]; self.cursor+=len(out); return out
    def state_dict(self): return {"n":self.n,"seed":self.seed,"epoch":self.epoch,"cursor":self.cursor}
    @classmethod
    def load(cls,s): return cls(**s)
sampler102=Sampler102(20,9); first102=sampler102.take(7); ss102=sampler102.state_dict(); continued102=sampler102.take(5); restored_sampler102=Sampler102.load(ss102)
assert np.array_equal(continued102,restored_sampler102.take(5))
assert len(set(first102)&set(continued102))==0
assert ss102["cursor"]==7 and ss102["epoch"]==0

## 3. 捕获与恢复完整 RNG 状态

checkpoint 应保存 Python `getstate`、NumPy state、PyTorch CPU RNG；使用 CUDA 时再保存 `torch.cuda.get_rng_state_all()`。恢复顺序很重要：对象初始化会消耗随机数，所以通常先构造对象与加载权重，再恢复训练 RNG。

In [ ]:
def rng_state102(): return {"python":random.getstate(),"numpy":np.random.get_state(),"torch":torch.get_rng_state().clone()}
def load_rng102(s): random.setstate(s["python"]); np.random.set_state(s["numpy"]); torch.set_rng_state(s["torch"])
seed_all102(123); saved_rng102=rng_state102(); draw1_102=(random.random(),np.random.rand(),torch.rand(4)); load_rng102(saved_rng102); draw2_102=(random.random(),np.random.rand(),torch.rand(4))
assert draw1_102[0]==draw2_102[0] and draw1_102[1]==draw2_102[1]
assert torch.equal(draw1_102[2],draw2_102[2])
assert saved_rng102["torch"].dtype==torch.uint8 and saved_rng102["torch"].ndim==1

## 4. 模型之外，还要保存 optimizer 与 scheduler

Adam 的动量、scheduler 的 step、AMP scaler 的 scale 都会影响下一次更新。checkpoint schema 还应含 global step、epoch、数据版本、代码/配置哈希。下面定义带 Dropout 的网络，让 RNG 丢失问题真正暴露出来。

In [ ]:
class Net102(nn.Module):
    def __init__(self): super().__init__(); self.w1=nn.Parameter(torch.randn(4,8)*.2); self.b1=nn.Parameter(torch.zeros(8)); self.w2=nn.Parameter(torch.randn(8,1)*.2); self.b2=nn.Parameter(torch.zeros(1)); self.drop=nn.Dropout(.25)
    def forward(self,x): return self.drop(torch.relu(x@self.w1+self.b1))@self.w2+self.b2
seed_all102(55); net_probe102=Net102(); opt_probe102=torch.optim.AdamW(net_probe102.parameters(),lr=.02); sched_probe102=torch.optim.lr_scheduler.StepLR(opt_probe102,step_size=3,gamma=.8)
state_keys102=set(opt_probe102.state_dict())
assert sum(p.numel() for p in net_probe102.parameters())==49
assert state_keys102=={"state","param_groups"}
assert sched_probe102.state_dict()["step_size"]==3

## 5. 连续训练与恢复训练的逐位回归测试

在第 5 步保存模型、AdamW、scheduler 与 RNG；原进程继续 5 步，另一对象从 checkpoint 恢复并执行相同 batch。因为 Dropout 消耗随机数，漏掉 RNG 时结果会分叉；完整恢复应使 loss、权重和 scheduler 完全一致。

In [ ]:
batches102=[torch.arange(i*6,(i+1)*6)%len(x102) for i in range(10)]
def train_step102(model,opt,sched,idx):
    pred=model(x102[idx]); loss=((pred-y102[idx])**2).mean(); opt.zero_grad(); loss.backward(); opt.step(); sched.step(); return float(loss.detach())
seed_all102(700); model_a102=Net102(); opt_a102=torch.optim.AdamW(model_a102.parameters(),lr=.015); sched_a102=torch.optim.lr_scheduler.StepLR(opt_a102,3,.8)
for idx in batches102[:5]: train_step102(model_a102,opt_a102,sched_a102,idx)
ckpt102={"model":copy.deepcopy(model_a102.state_dict()),"optimizer":copy.deepcopy(opt_a102.state_dict()),"scheduler":copy.deepcopy(sched_a102.state_dict()),"rng":rng_state102(),"step":5}
loss_a102=[train_step102(model_a102,opt_a102,sched_a102,idx) for idx in batches102[5:]]
model_b102=Net102(); opt_b102=torch.optim.AdamW(model_b102.parameters(),lr=.015); sched_b102=torch.optim.lr_scheduler.StepLR(opt_b102,3,.8); model_b102.load_state_dict(ckpt102["model"]); opt_b102.load_state_dict(ckpt102["optimizer"]); sched_b102.load_state_dict(ckpt102["scheduler"]); load_rng102(ckpt102["rng"])
loss_b102=[train_step102(model_b102,opt_b102,sched_b102,idx) for idx in batches102[5:]]
assert loss_a102==loss_b102
assert all(torch.equal(a,b) for a,b in zip(model_a102.state_dict().values(),model_b102.state_dict().values()))
assert sched_a102.state_dict()==sched_b102.state_dict() and ckpt102["step"]==5

## 6. 半个梯度累积窗口如何保存

最简单可靠的策略是在 optimizer step 边界落 checkpoint。若必须在半窗口保存，仅有模型权重不够，还需每个参数当前 `.grad`、已累计有效样本数、micro-step 和 scaler 状态；否则恢复后有效 batch 或梯度尺度改变。

In [ ]:
class AccumState102:
    def __init__(self): self.grad=np.zeros(3); self.samples=0; self.micro_step=0
    def add(self,grad_sum,n): self.grad+=np.asarray(grad_sum); self.samples+=n; self.micro_step+=1
    def state_dict(self): return {"grad":self.grad.copy(),"samples":self.samples,"micro_step":self.micro_step}
    @classmethod
    def load(cls,s): obj=cls(); obj.grad=s["grad"].copy(); obj.samples=s["samples"]; obj.micro_step=s["micro_step"]; return obj
acc102=AccumState102(); acc102.add([2.,4.,6.],2); mid102=acc102.state_dict(); acc102.add([3.,3.,3.],3); resumed_acc102=AccumState102.load(mid102); resumed_acc102.add([3.,3.,3.],3)
assert np.array_equal(acc102.grad,resumed_acc102.grad)
assert np.allclose(acc102.grad/acc102.samples,[1.,1.4,1.8])
assert resumed_acc102.samples==5 and resumed_acc102.micro_step==2

## 7. DataLoader worker 也需要独立且确定的 seed

多 worker 数据增强若共享或复制 RNG，会产生重复增强；若 seed 只依赖 worker ID，每个 epoch 又会重复。可由 run seed、epoch、rank、worker ID 通过稳定哈希派生 seed，禁止使用进程随机化的 Python `hash()`。

In [ ]:
def worker_seed102(run_seed,epoch,rank,worker):
    raw=f"{run_seed}:{epoch}:{rank}:{worker}".encode(); return int.from_bytes(hashlib.sha256(raw).digest()[:8],"little")%(2**32)
ws_a102=worker_seed102(7,2,0,3); ws_b102=worker_seed102(7,2,0,3); ws_other102=worker_seed102(7,3,0,3)
aug_a102=np.random.default_rng(ws_a102).normal(size=5); aug_b102=np.random.default_rng(ws_b102).normal(size=5)
assert ws_a102==ws_b102 and ws_a102!=ws_other102
assert np.array_equal(aug_a102,aug_b102)
assert 0<=ws_a102<2**32

## 8. Checkpoint 完整性、兼容性与复现等级

保存到临时文件、fsync 后原子 rename，避免进程中断留下半文件；同时记录 checksum。加载前校验 schema、模型签名、数据快照、软件/硬件版本。GPU 非确定 kernel 或跨版本库可能无法逐位一致，应明确承诺“bitwise / tolerance / statistical”哪一级。

In [ ]:
publish102={"schema":1,"step":ckpt102["step"],"model":ckpt102["model"],"optimizer":ckpt102["optimizer"],"scheduler":ckpt102["scheduler"],"rng":ckpt102["rng"],"manifest":{"data":"train-v3","code":"commit-example","reproducibility":"same_stack_bitwise"}}
buf102=io.BytesIO(); torch.save(publish102,buf102); payload102=buf102.getvalue(); checksum102=hashlib.sha256(payload102).hexdigest(); buf102.seek(0); loaded102=torch.load(buf102,weights_only=False)
assert len(payload102)>1000 and len(checksum102)==64
assert loaded102["step"]==5 and loaded102["manifest"]["data"]=="train-v3"
assert set(loaded102)>= {"schema","model","optimizer","scheduler","rng","manifest"}

## 面试总结

完整答案是 **定义复现等级 → 隔离所有 RNG → 保存 sampler/cursor → 保存 model/optimizer/scheduler/scaler → 保存累积窗口 → 恢复顺序 → 连续/恢复逐位测试 → 原子文件与 manifest**。仅说 `manual_seed`，无法解释 Dropout、数据顺序、Adam 动量和半窗口恢复，因此不够工程化。

延伸阅读：[PyTorch Reproducibility](https://pytorch.org/docs/stable/notes/randomness.html)、[Saving and Loading Models](https://pytorch.org/tutorials/beginner/saving_loading_models.html)、[DataLoader 文档](https://pytorch.org/docs/stable/data.html)。